In [0]:
import json
import os
from datetime import datetime
import random
from openai import OpenAI

# =============================================================================
# OpenAI Connection Setup via Databricks Secrets
# =============================================================================

try:
    SECRET_SCOPE = "openai"
    API_KEY_NAME = "GDPR_agent"
    
    api_key = dbutils.secrets.get(scope=SECRET_SCOPE, key=API_KEY_NAME)
    openai_client = OpenAI(api_key=api_key)
    print("✅ OpenAI client initialized successfully from dbutils secrets.")
except Exception as e:
    print(f"❌ Failed to initialize OpenAI client from secrets: {e}")
    openai_client = None

# =============================================================================
# LLM-Based Question Generation
# =============================================================================

def generate_unbiased_question(target_concept: str, source_context: str, question_type: str = "single") -> str:
    """Generate natural user question avoiding literal concept mention."""
    if not openai_client:
        return generate_fallback_question(target_concept, source_context, question_type)

    if question_type == "single":
        system_prompt = f"""You are generating evaluation test questions for a GDPR compliance assistant.

CRITICAL CONSTRAINT: Generate a realistic, natural user question about the core themes or requirements covered in '{target_concept}' inside the context of {source_context}. 
You MUST NOT use the literal phrase or words from '{target_concept}' anywhere in your question.

Instead, describe what this concept covers using:
- Synonyms (e.g., "erasing client profiles" instead of "right to erasure")
- Contextual situations (e.g., "what happens when a user demands we remove their data history")
- General compliance language.

The question must sound like an executive or non-technical business user asking for help. 
Generate ONE question sentence only. Do not add intro text, bullet points, or quotes."""

    elif question_type == "multi":
        system_prompt = f"""You are generating an evaluation test question for a GDPR compliance assistant.

CRITICAL CONSTRAINT: Generate a natural user scenario that requires cross-referencing information from multiple sources about '{target_concept}'. 
You MUST NOT use the literal text or words from '{target_concept}' anywhere in your question.

Formulate a multi-part user inquiry that naturally bridges compliance frameworks and empirical tracking. 
Example pattern: "What are the rules regarding notification periods for breaches, and what kind of enforcement penalties have companies run into for dragging their feet?"

Generate ONE question sentence only. Do not add quotes or markdown syntax."""
    
    else:
        system_prompt = f"""Generate a natural user question about '{target_concept}' in {source_context} context, avoiding literal mention of '{target_concept}'."""

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",  # Use cheaper model for generation
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": "Generate the question now."}
            ],
            temperature=0.9,  # Higher temperature for more variety
            max_tokens=150
        )
        
        if response.choices and len(response.choices) > 0:
            question = response.choices[0].message.content.strip().strip('"').strip("'")
            
            # Validation safeguard
            if target_concept.lower() in question.lower():
                print(f"⚠️  LLM included literal '{target_concept}' - fallback triggered")
                return generate_fallback_question(target_concept, source_context, question_type)
            return question
        return generate_fallback_question(target_concept, source_context, question_type)
            
    except Exception as e:
        print(f"⚠️  OpenAI LLM call failed ({e}) - using fallback")
        return generate_fallback_question(target_concept, source_context, question_type)


def generate_fallback_question(target_concept: str, source_context: str, question_type: str) -> str:
    """Fallback questions if LLM fails."""
    if question_type == "single":
        if "Article" in target_concept or "Context" in target_concept:
            return f"What are our baseline transparent communication obligations when handling requests for access to personal information details under {source_context}?"
        elif source_context == "fines":
            return "What major financial penalties have European regulators issued for compliance tracking failures?"
        return "What is our company baseline position on retention timelines for standard behavioral tracking data?"
    return "What are the strict operational notification limits for security incidents, and what fines have been levied for late reports?"


# =============================================================================
# Source Table Configuration
# =============================================================================

print("🔍 GENERATING 200 UNIQUE QUESTIONS FROM SOURCE DATA")
print("="*80)

LEGISLATION_SOURCE_TABLE = "main.default.gdpr_statutory_legislation" 
FINES_SOURCE_TABLE = "main.default.gdpr_translated_chunks"     
POLICY_SOURCE_TABLE = "main.default.retail_corporate_policy"

# =============================================================================
# Data Profiling (FIX: Move count() inside try block before toPandas)
# =============================================================================

# Legislation Data Preparation
legislation_articles = []
try:
    legislation_df = spark.table(LEGISLATION_SOURCE_TABLE).limit(500)
    row_count = legislation_df.count()  # Trigger action FIRST
    legislation_df = legislation_df.toPandas()
    if 'article_title' in legislation_df.columns:
        legislation_articles = [a for a in legislation_df['article_title'].dropna().unique().tolist() if a]
    print(f"✅ Loaded {len(legislation_articles)} legislation articles")
except Exception as e:
    print(f"❌ Legislation read error: {e}")
    legislation_articles = ["General Modalities Context", "Right of access", "Right to rectification"]

# Fines Data Preparation
companies_found = []
try:
    fines_df = spark.table(FINES_SOURCE_TABLE).limit(500)
    row_count = fines_df.count()  # Trigger action FIRST
    fines_df = fines_df.toPandas()
    companies_set = set()
    if 'translated_text' in fines_df.columns:
        for text in fines_df['translated_text'].dropna():
            for c in ['british airways', 'google', 'amazon', 'meta', 'facebook', 'marriott', 'whatsapp', 'uber', 'tiktok', 'twitter']:
                if c in str(text).lower(): 
                    companies_set.add(c.title())
    companies_found = sorted(list(companies_set))
    print(f"✅ Loaded {len(companies_found)} companies from fines data")
except Exception as e:
    print(f"❌ Fines read error: {e}")
    companies_found = ["Facebook", "Google", "British Airways"]

# Policy Data Preparation
policy_sections = []
try:
    policy_df = spark.table(POLICY_SOURCE_TABLE).limit(500)
    row_count = policy_df.count()  # Trigger action FIRST
    policy_df = policy_df.toPandas()
    if 'section_title' in policy_df.columns:
        policy_sections = [s for s in policy_df['section_title'].dropna().unique().tolist() if s]
    print(f"✅ Loaded {len(policy_sections)} policy sections")
except Exception as e:
    print(f"❌ Policy read error: {e}")
    policy_sections = ["Global Policy Overview", "Customer Identity Profile", "Data Retention"]

# Fallback populations
if not legislation_articles: 
    legislation_articles = ["General Modalities Context", "Right of access", "Right to rectification"]
if not companies_found: 
    companies_found = ["Facebook", "Google", "British Airways"]
if not policy_sections: 
    policy_sections = ["Global Policy Overview", "Customer Identity Profile", "Data Retention"]

# =============================================================================
# Generate 200 Questions with Distribution
# =============================================================================

print("\n" + "="*80)
print("🎯 GENERATING 200 UNIQUE QUESTIONS")
print("="*80)

test_cases = []

# Distribution: 200 questions
# 80 legislation, 80 fines, 30 policy, 10 multi-source
mix_config = [
    {"count": 80, "cat": "single_source_legislation", "ctx": "GDPR legislation", "src": ["legislation"], "type": "single"},
    {"count": 80, "cat": "single_source_fines", "ctx": "GDPR enforcement cases", "src": ["fines"], "type": "single"},
    {"count": 30, "cat": "single_source_policy", "ctx": "internal corporate policy", "src": ["policy"], "type": "single"},
    {"count": 10, "cat": "multi_source_fines_legislation", "ctx": "legislation and fines", "src": ["fines", "legislation"], "type": "multi"}
]

global_id = 1

for config in mix_config:
    print(f"\n📂 Generating {config['count']} questions for {config['cat']}...")
    
    for idx in range(config["count"]):
        case_id = f"q_{global_id:04d}"
        
        if config["cat"] == "single_source_legislation":
            concept = legislation_articles[idx % len(legislation_articles)]
            expected = {"sources": config["src"], "must_retrieve_from_articles": [concept]}
            data_src = LEGISLATION_SOURCE_TABLE
            
        elif config["cat"] == "single_source_fines":
            concept = companies_found[idx % len(companies_found)] if idx > 0 else "largest GDPR fines"
            expected = {"sources": config["src"], "must_retrieve_keywords": [concept.lower(), "fine"]}
            data_src = FINES_SOURCE_TABLE
            
        elif config["cat"] == "single_source_policy":
            concept = policy_sections[idx % len(policy_sections)]
            expected = {"sources": config["src"], "must_retrieve_from_sections": [concept]}
            data_src = POLICY_SOURCE_TABLE
            
        elif config["cat"] == "multi_source_fines_legislation":
            leg_concept = legislation_articles[idx % len(legislation_articles)]
            expected = {
                "sources": config["src"],
                "legislation_check": [leg_concept],
                "fines_check": ["fine", "breach"]
            }
            concept = f"{leg_concept} compliance alignment"
            data_src = f"{LEGISLATION_SOURCE_TABLE} & {FINES_SOURCE_TABLE}"

        # Generate question
        question_text = generate_unbiased_question(
            target_concept=concept, 
            source_context=config["ctx"], 
            question_type=config["type"]
        )
        
        if global_id % 20 == 0:  # Progress indicator every 20 questions
            print(f"   [{global_id}/200] Generated: {question_text[:60]}...")

        test_cases.append({
            "question_id": case_id,
            "question": question_text,
            "category": config["cat"],
            "source_category": config["src"][0],  # Primary source
            "concept": concept,
            "difficulty": "medium" if global_id % 3 == 0 else "easy",
            "expected_sources": str(config["src"]),
            "data_source_table": data_src,
            "generation_method": "llm_paraphrased",
            "created_date": datetime.now()
        })
        
        global_id += 1

print(f"\n✅ Generated {len(test_cases)} questions")

# =============================================================================
# Save to Delta Table
# =============================================================================

from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Define schema
schema = StructType([
    StructField("question_id", StringType(), False),
    StructField("question", StringType(), False),
    StructField("category", StringType(), False),
    StructField("source_category", StringType(), False),
    StructField("concept", StringType(), True),
    StructField("difficulty", StringType(), True),
    StructField("expected_sources", StringType(), True),
    StructField("data_source_table", StringType(), True),
    StructField("generation_method", StringType(), True),
    StructField("created_date", TimestampType(), False)
])

# Create DataFrame
questions_df = spark.createDataFrame(test_cases, schema)

# Save to Delta
QUESTION_POOL_TABLE = "main.default.gdpr_agent_question_pool"
questions_df.write.mode("overwrite").saveAsTable(QUESTION_POOL_TABLE)

print("\n" + "="*80)
print(f"🚀 SUCCESS: Question pool saved to {QUESTION_POOL_TABLE}")
print(f"📊 Total Questions: {questions_df.count()}")
print("="*80)

# Show sample
print("\n📋 Sample Questions:")
display(questions_df.orderBy("question_id").limit(10))

# Category distribution
print("\n📊 Distribution by Category:")
display(questions_df.groupBy("category").count().orderBy("count", ascending=False))